# Explorative Datenanalyse (EDA) mit „modernen“ EDA-Bibliotheken (1 Stunde)  
**VHS – Tag 1 | Nach Datenbereinigung & Datenvalidierung**

Heute machen wir EDA **schnell, visuell und strukturiert** – mit klassischen Tools (Pandas/Matplotlib) **und** modernen EDA-Libraries wie:
- **ydata-profiling** (automatischer Data Bericht)
- **sweetviz** (vergleichende EDA-Berichts)
- **plotly** (interaktive Visualisierungen)
- optional: **missingno** (fehlende Werte Visuals)

> Hinweis: In manchen Umgebungen (VHS-PC, Firmenlaptop) funktionieren nicht alle Libraries sofort.  
> Deshalb: Wir starten **robust** (Pandas + Matplotlib) und nutzen die neuen Libraries **als Turbo**, wenn Installation möglich ist.


## Lernziele (am Ende der Stunde könnt ihr…)
- einen Datensatz schnell überblicken (Shape, Typen, fehlende Werte)
- zentrale Verteilungen und Ausreißer erkennen
- einfache Segmentierungen vergleichen (z. B. pro Kurs)
- automatisch einen EDA-Bericht generieren (ydata-profiling / sweetviz)


## 0) Setup
Wir nutzen denselben Demo-Datensatz wie bei der Validierung (Anmeldedaten).  
Wenn du schon ein `df` aus dem Validierungs-Notebook hast, kannst du diesen Teil überspringen.


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

rng = np.random.default_rng(42)


In [ ]:
# Demo-Daten (mit ein paar eingebauten Problemen) – damit EDA „realistisch“ ist.
def make_demo_data(n=200, seed=42):
    rng = np.random.default_rng(seed)
    kurs = rng.choice(["Python-Grundlagen", "Datenvisualisierung", "DL-Einführung"], size=n, replace=True, p=[0.45, 0.35, 0.2])
    zahlungsstatus = rng.choice(["bezahlt", "offen", "storniert"], size=n, replace=True, p=[0.65, 0.25, 0.10])

    df = pd.DataFrame({
        "teilnehmer_id": [f"T{1000+i}" for i in range(n)],
        "name": rng.choice(["Ali", "Mia", "Noah", "Lina", "Ben", "Sara", "Omar", "Lea"], size=n),
        "email": [f"user{i}@example.com" for i in range(n)],
        "alter": rng.integers(16, 65, size=n),
        "kurs": kurs,
        "kursdatum": pd.to_datetime("2026-03-10") + pd.to_timedelta(rng.integers(-10, 20, size=n), unit="D"),
        "betrag_eur": np.round(rng.normal(79, 18, size=n), 2),
        "zahlungsstatus": zahlungsstatus,
        "stunden": rng.integers(1, 9, size=n)
    })

    # absichtliche „Datenrealität“
    df.loc[rng.choice(df.index, 5, replace=False), "alter"] = rng.choice([-3, 150], size=5)  # unplausibel
    df.loc[rng.choice(df.index, 6, replace=False), "email"] = "not-an-email"                # falsches Format
    df.loc[rng.choice(df.index, 3, replace=False), "kurs"] = "Data Vizz"                    # Tippfehler
    df.loc[rng.choice(df.index, 4, replace=False), "betrag_eur"] = -10                      # unplausibel
    df.loc[rng.choice(df.index, 4, replace=False), "stunden"] = np.nan                      # Missing
    df.loc[rng.choice(df.index, 2, replace=False), "name"] = None                           # Missing
    df.loc[rng.choice(df.index, 2, replace=False), "betrag_eur"] = "79 EUR"                 # falscher Typ
    
    return df

df = make_demo_data()
df.head()


## 1) Datenüberblick (5–10 Min)

Ziel: **Schnell** verstehen:
- Wie groß ist der Datensatz?
- Welche Spalten & Typen?
- Gibt es fehlende Werte?
- Gibt es offensichtliche Probleme?


In [ ]:
df.shape, df.columns.tolist()


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


In [ ]:
# fehlende Werte pro Spalte
df.isna().sum().sort_values(ascending=False)


## 2) „Schneller Fix“ für EDA (ohne Bereinigung-Overkill) (5 Min)

Damit Plots nicht crashen:
- Zahlen-Spalten einmal in numerisch konvertierenn
- Datum konvertierenn
- Kategorien als string behandeln


In [ ]:
df_eda = df.copy()

df_eda["betrag_eur"] = pd.to_numeric(df_eda["betrag_eur"], errors="konvertieren")
df_eda["alter"] = pd.to_numeric(df_eda["alter"], errors="konvertieren")
df_eda["stunden"] = pd.to_numeric(df_eda["stunden"], errors="konvertieren")
df_eda["kursdatum"] = pd.to_datetime(df_eda["kursdatum"], errors="konvertieren")

df_eda.head()


## 3) Univariate EDA: Verteilungen & Ausreißer (15 Min)

Wir schauen uns einzelne Variablen an:
- numerisch: Histogramm, Boxplot
- kategorisch: Wertehäufigkeiten / Balken


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
# Histogramm: Alter
plt.figure(figsize=(7,4))
plt.hist(df_eda["alter"].dropna(), bins=20)
plt.title("Verteilung: Alter")
plt.xlabel("Alter")
plt.ylabel("Anzahl")
plt.show()


In [ ]:
# Boxplot: Betrag
plt.figure(figsize=(7,4))
plt.boxplot(df_eda["betrag_eur"].dropna(), vert=False)
plt.title("Boxplot: Betrag (EUR)")
plt.xlabel("EUR")
plt.show()


In [ ]:
# Kategorien: Kurs
df_eda["kurs"].value_counts(dropna=False)


In [ ]:
plt.figure(figsize=(7,4))
df_eda["kurs"].value_counts(dropna=False).plot(kind="bar")
plt.title("Häufigkeit: Kurs")
plt.xlabel("Kurs")
plt.ylabel("Anzahl")
plt.show()


## 4) Bivariate EDA: Beziehungen vergleichen (10–15 Min)

Beispiele:
- Betrag nach Kurs
- Alter nach Zahlungsstatus
- Stunden nach Kurs


In [ ]:
# Gruppierte Kennzahlen: Betrag nach Kurs
df_eda.groupby("kurs")["betrag_eur"].agg(["count", "mean", "median", "min", "max"]).sort_values("mean", ascending=False)


In [ ]:
# Boxplots: Betrag nach Kurs (klassisch, robust)
kurs_list = [k for k in df_eda["kurs"].dropna().unique()]
data = [df_eda.loc[df_eda["kurs"] == k, "betrag_eur"].dropna() for k in kurs_list]

plt.figure(figsize=(9,4))
plt.boxplot(data, labels=kurs_list, vert=False)
plt.title("Betrag nach Kurs")
plt.xlabel("EUR")
plt.show()


## 5) Moderne EDA-Libraries (15–20 Min)
### 5.1 Installation (falls nötig)

Wenn ihr in Colab oder eigener Umgebung seid, könnt ihr installieren.  
Auf manchen VHS-Rechnern sind Installationen evtl. eingeschränkt – dann überspringen.


In [ ]:
# OPTIONAL: Nur ausführen, wenn Installation erlaubt ist.
# In Colab: funktioniert meist.
# !pip -q install ydata-profiling sweetviz plotly missingno


### 5.2 ydata-profiling: Automatischer Profiling-Bericht (1 Befehl)

Ergebnis: HTML-Bericht mit:
- fehlende Werte
- Verteilungen
- Korrelationen
- „Warnungen“ (z. B. hohe Kardinalität, mögliche ID-Spalten, etc.)


In [ ]:
# OPTIONAL: ydata-profiling Bericht (HTML)
# Wenn es import-Fehler gibt: Installation überspringen oder nur klassische EDA nutzen.

# from ydata_profiling import ProfileBericht
# profile = ProfileBericht(df_eda, title="EDA Bericht – Kursdaten", explorative=True)
# profile.to_file("eda_report_ydata_profiling.html")
# print("Bericht gespeichert: eda_report_ydata_profiling.html")


### 5.3 sweetviz: Schneller EDA-Bericht + Vergleich (z. B. bezahlt vs offen)

Sweetviz ist praktisch, um **zwei Gruppen** direkt zu vergleichen.


In [ ]:
# OPTIONAL: Sweetviz Bericht (HTML)
# import sweetviz as sv

# report = sv.analyze(df_eda)
# report.show_html("eda_report_sweetviz.html", open_browser=False)
# print("Bericht gespeichert: eda_report_sweetviz.html")

# BONUS: Vergleich (bezahlt vs offen)
# df_paid = df_eda[df_eda["zahlungsstatus"] == "bezahlt"]
# df_open = df_eda[df_eda["zahlungsstatus"] == "offen"]
# comp = sv.compare([df_paid, "bezahlt"], [df_open, "offen"])
# comp.show_html("eda_compare_bezahlt_vs_offen.html", open_browser=False)
# print("Vergleich gespeichert: eda_compare_bezahlt_vs_offen.html")


### 5.4 Plotly: Interaktive Plots (praktisch für den Live-Unterricht)

Wenn Plotly verfügbar ist, bekommt ihr Zoom/Tooltip/Filter.


In [ ]:
# OPTIONAL: Plotly (interaktiv)
# import plotly.express as px
# fig = px.histogram(df_eda, x="betrag_eur", nbins=30, title="Interaktiv: Betrag (EUR)")
# fig.show()


## 6) Mini-Übung (10 Min): „3 Erkenntnisse in 10 Minuten“

Aufgabe:
1) Findet **eine Auffälligkeit** (Ausreißer / Tippfehler / Missing)  
2) Findet **eine Verteilung**, die euch überrascht  
3) Findet **einen Vergleich**, der interessant ist (z. B. Betrag nach Kurs)

Schreibt eure 3 Erkenntnisse als Stichpunkte:

- Erkenntnis 1: … (Beleg: welche Tabelle/Plot?)  
- Erkenntnis 2: …  
- Erkenntnis 3: …  


In [ ]:
# Schreibraum für eure 3 Erkenntnisse (hier als Textliste)
insights = [
    "Erkenntnis 1: ...",
    "Erkenntnis 2: ...",
    "Erkenntnis 3: ..."
]
insights


## Abschluss (1–2 Min)

- EDA ist **ein Dialog mit den Daten**: anschauen, fragen, prüfen, vergleichen.
- Moderne EDA-Libraries liefern schnell einen **Qualitäts- und Struktur-Überblick**.
- Aber: Die besten Erkenntnisse kommen oft aus **einfachen, gezielten** Plots & Gruppierungen.

➡️ Nächster Kurs-Schritt (optional):  
- Feature Engineering / Vorbereitung fürs Modell (Skalierung, Encoding)  
- oder direkt: erstes kleines ML/DL-Modell
